# 03 — Merge ACLED + GDELT

This notebook merges the preprocessed ACLED country-week panel with the preprocessed GDELT media panel into a single dataset for the extended model.

| | |
|---|---|
| Input 1 | `processed/acled_country_week.csv` |
| Input 2 | `processed/gdelt_country_week_processed.csv` |
| Output | `processed/merged_panel.csv` |

---

## Pipeline

1. Load both processed datasets
2. Validate alignment (same countries, same weeks, Monday-anchored)
3. Left-merge on `[COUNTRY, week_start]` — ACLED is the backbone
4. Handle any remaining NaN in GDELT columns
5. Final diagnostics
6. Export

## Why a left merge?

ACLED is the primary dataset: it defines the country-weeks for which we have conflict outcome data. A left merge keeps every ACLED row and attaches GDELT media data where available. If GDELT has no record for a particular country-week, the GDELT columns will be NaN and are filled with country means below.

## 0. Setup

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

# Paths relative to repo root
REPO_ROOT = Path().resolve().parent  # one level up from notebooks/

PROC_DIR  = REPO_ROOT / "processed"

PROC_DIR.mkdir(parents=True, exist_ok=True)

ACLED_PATH  = PROC_DIR / "acled_country_week.csv"
GDELT_PATH  = PROC_DIR / "gdelt_country_week_processed.csv"
MERGED_PATH = PROC_DIR / "merged_panel.csv"

TARGET_COUNTRIES = ["Mali", "Niger", "Nigeria", "Burkina Faso", "Ghana"]

print(f"ACLED path exists:  {ACLED_PATH.exists()}")
print(f"GDELT path exists:  {GDELT_PATH.exists()}")

ACLED path exists:  True
GDELT path exists:  True


## 1. Load Processed Data

In [2]:
acled = pd.read_csv(ACLED_PATH, parse_dates=["week_start"])
gdelt = pd.read_csv(GDELT_PATH, parse_dates=["week_start"])

print(f"ACLED shape: {acled.shape}")
print(f"GDELT shape: {gdelt.shape}")
print("\nACLED columns:", acled.columns.tolist())
print("\nGDELT columns:", gdelt.columns.tolist())

ACLED shape: (2345, 26)
GDELT shape: (2345, 10)

ACLED columns: ['COUNTRY', 'week_start', 'n_events', 'n_fatalities', 'events_lag1', 'events_lag2', 'events_lag3', 'events_lag4', 'fatalities_lag1', 'fatalities_lag2', 'events_ma4', 'events_ma8', 'events_ma12', 'events_std4', 'events_std8', 'events_delta4', 'conflict_lag1', 'conflict_lag2', 'conflict_weeks_last4', 'conflict_weeks_last8', 'events_h1', 'conflict_h1', 'events_h2', 'conflict_h2', 'events_h4', 'conflict_h4']

GDELT columns: ['COUNTRY', 'week_start', 'n_events_gdelt', 'total_mentions', 'total_articles', 'avg_tone', 'avg_goldstein', 'tone_lag1', 'goldstein_lag1', 'mentions_lag1']


## 2. Validate Alignment

Both datasets must have the same countries, overlapping date ranges, and Monday-anchored week starts before merging.

In [3]:
# Country overlap
acled_c = set(acled["COUNTRY"].unique())
gdelt_c = set(gdelt["COUNTRY"].unique())
print("Countries in ACLED:", sorted(acled_c))
print("Countries in GDELT:", sorted(gdelt_c))
print("Only in ACLED:", acled_c - gdelt_c)
print("Only in GDELT:", gdelt_c - acled_c)

# Date ranges
print(f"\nACLED weeks: {acled['week_start'].min().date()} → {acled['week_start'].max().date()}")
print(f"GDELT weeks: {gdelt['week_start'].min().date()} → {gdelt['week_start'].max().date()}")

# Monday alignment
assert (acled["week_start"].dt.weekday == 0).all(), "ACLED has non-Monday weeks"
assert (gdelt["week_start"].dt.weekday == 0).all(), "GDELT has non-Monday weeks"
print("\nBoth datasets are Monday-aligned ✓")

Countries in ACLED: ['Burkina Faso', 'Ghana', 'Mali', 'Niger', 'Nigeria']
Countries in GDELT: ['Burkina Faso', 'Ghana', 'Mali', 'Niger', 'Nigeria']
Only in ACLED: set()
Only in GDELT: set()

ACLED weeks: 2015-01-05 → 2023-12-25
GDELT weeks: 2015-01-05 → 2023-12-25

Both datasets are Monday-aligned ✓


## 3. Merge

Select only the GDELT columns needed for the model and the merge keys. The raw (unlagged) media values are kept as reference columns; only the lagged versions enter the regression.

In [4]:
# GDELT columns to bring into the merged panel
GDELT_COLS = [
    "COUNTRY", "week_start",
    # Raw values — kept for reference and descriptive statistics
    "avg_tone",
    "avg_goldstein",
    "total_mentions",
    "total_articles",
    # Lagged model predictors — these enter the regression
    "tone_lag1",
    "goldstein_lag1",
    "mentions_lag1",
]

gdelt_cols_present = [c for c in GDELT_COLS if c in gdelt.columns]
missing = set(GDELT_COLS) - set(gdelt_cols_present)
if missing:
    print(f"WARNING: GDELT columns not found: {missing}")

gdelt_slim = gdelt[gdelt_cols_present].copy()

# Left merge — keeps all ACLED rows
merged = acled.merge(gdelt_slim, on=["COUNTRY", "week_start"], how="left",
                     suffixes=("_acled", "_gdelt"))

print(f"Merged shape: {merged.shape}")

# A left merge should never add rows
assert len(merged) == len(acled), (
    f"Merge created {len(merged) - len(acled)} extra rows — "
    "check for duplicate keys in the GDELT panel"
)
print("No duplicate rows created ✓")

Merged shape: (2345, 33)
No duplicate rows created ✓


## 4. Handle Post-Merge NaN Values

NaN in GDELT columns after the merge means there was no GDELT record for that country-week. These are filled with the country-level mean for each column, which is a neutral imputation that does not bias the predictor distribution.

In [5]:
gdelt_predictor_cols = [
    c for c in merged.columns
    if c not in acled.columns and c not in ["COUNTRY", "week_start"]
]

print("NaN in GDELT columns after merge:")
na = merged[gdelt_predictor_cols].isna().sum()
print(na[na > 0] if na.any() else "  None ✓")

# Fill with country mean
for col in gdelt_predictor_cols:
    if merged[col].isna().any():
        merged[col] = merged.groupby("COUNTRY")[col].transform(
            lambda x: x.fillna(x.mean())
        )

# Fill any remaining NaN with global mean
for col in gdelt_predictor_cols:
    if merged[col].isna().any():
        merged[col] = merged[col].fillna(merged[col].mean())

print("NaN after imputation:")
na_after = merged[gdelt_predictor_cols].isna().sum()
print(na_after[na_after > 0] if na_after.any() else "  None ✓")

NaN in GDELT columns after merge:
  None ✓
NaN after imputation:
  None ✓


## 5. Final Diagnostics

In [6]:
print(f"Final merged panel shape: {merged.shape}")
print(f"Countries: {sorted(merged['COUNTRY'].unique())}")
print(f"Date range: {merged['week_start'].min().date()} → {merged['week_start'].max().date()}")
print(f"\nWeeks per country:")
print(merged.groupby("COUNTRY")["week_start"].count())

print("\nKey GDELT column ranges:")
for col, (lo, hi) in {"avg_tone": (-100, 0), "avg_goldstein": (-10, 0),
                      "tone_lag1": (-100, 0), "goldstein_lag1": (-10, 0)}.items():
    if col in merged.columns:
        print(f"  {col}: [{merged[col].min():.2f}, {merged[col].max():.2f}]")

Final merged panel shape: (2345, 33)
Countries: ['Burkina Faso', 'Ghana', 'Mali', 'Niger', 'Nigeria']
Date range: 2015-01-05 → 2023-12-25

Weeks per country:
COUNTRY
Burkina Faso    469
Ghana           469
Mali            469
Niger           469
Nigeria         469
Name: week_start, dtype: int64

Key GDELT column ranges:
  avg_tone: [-90.95, -13.75]
  avg_goldstein: [-10.00, -3.50]
  tone_lag1: [-90.95, 0.00]
  goldstein_lag1: [-10.00, 0.00]


In [7]:
# Correlation between key ACLED and GDELT predictors
# Useful for understanding relationships before modelling
check_cols = [
    c for c in [
        "events_lag1", "events_ma4", "events_std4", "conflict_weeks_last8",
        "tone_lag1", "goldstein_lag1", "mentions_lag1",
    ] if c in merged.columns
]

corr = merged[check_cols].corr().round(3)
print("Pairwise correlation between ACLED and GDELT predictors:")
print(corr.to_string())

Pairwise correlation between ACLED and GDELT predictors:
                      events_lag1  events_ma4  events_std4  conflict_weeks_last8  tone_lag1  goldstein_lag1  mentions_lag1
events_lag1                 1.000       0.970        0.743                 0.422     -0.373          -0.419         -0.328
events_ma4                  0.970       1.000        0.763                 0.436     -0.393          -0.437         -0.337
events_std4                 0.743       0.763        1.000                 0.443     -0.374          -0.426         -0.288
conflict_weeks_last8        0.422       0.436        0.443                 1.000     -0.317          -0.365         -0.180
tone_lag1                  -0.373      -0.393       -0.374                -0.317      1.000           0.759          0.050
goldstein_lag1             -0.419      -0.437       -0.426                -0.365      0.759           1.000          0.234
mentions_lag1              -0.328      -0.337       -0.288                -0.180  

## 6. Export

In [8]:
merged.to_csv(MERGED_PATH, index=False)
print(f"Saved → {MERGED_PATH}")
print(f"Shape: {merged.shape}")
print(f"Columns: {merged.columns.tolist()}")

Saved → C:\Users\faiza\Documents\Uni\Year 3\Thesis\Data Preprocessing\processed\merged_panel.csv
Shape: (2345, 33)
Columns: ['COUNTRY', 'week_start', 'n_events', 'n_fatalities', 'events_lag1', 'events_lag2', 'events_lag3', 'events_lag4', 'fatalities_lag1', 'fatalities_lag2', 'events_ma4', 'events_ma8', 'events_ma12', 'events_std4', 'events_std8', 'events_delta4', 'conflict_lag1', 'conflict_lag2', 'conflict_weeks_last4', 'conflict_weeks_last8', 'events_h1', 'conflict_h1', 'events_h2', 'conflict_h2', 'events_h4', 'conflict_h4', 'avg_tone', 'avg_goldstein', 'total_mentions', 'total_articles', 'tone_lag1', 'goldstein_lag1', 'mentions_lag1']
